<a href="https://colab.research.google.com/github/gramucse/ai-mentor-portfolio/blob/main/Day6_PlacementProcessor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 6 — Placement Data Processor (Capstone Sprint 1)**Goal:** JD URL → BeautifulSoup → Gemini structured call → clean JSON.**Time:** 90 min sprint after the 90 min Lab 6A.

## Step 1 — Install + key

In [1]:
!pip install -q google-genai pydantic

import os
import getpass

if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


## Step 2 — JD Pydantic schema

In [2]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

## Step 3 — Scraper + normaliserIf live scraping is blocked, fall back to `../data/jds_cached.jsonl`.

In [3]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    """Extract a Resume JSON from raw text. Retries once on schema fail."""
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            fix_prompt = f'Fix this JSON to match schema. Errors: {e}. Original: {resp.text}'
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={'response_mime_type': 'application/json',
                        'response_schema': Resume.model_json_schema()})
            return Resume.model_validate_json(resp.text)

## Step 4 — Process 5 JDs (or load cached)

In [5]:
with open('./sample_data/sample_resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]
print(f'Loaded {len(resumes)} sample résumés')

results = []
errors = []
for i, r in enumerate(resumes):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'  [{i+1}] {parsed.name} — {len(parsed.skills)} skills')
    except Exception as e:
        errors.append((i, e))
        print(f'  [{i+1}] FAILED: {type(e).__name__}: {str(e)[:120]}')

print(f'\n{len(results)}/5 succeeded, {len(errors)} failed')

Loaded 5 sample résumés
  [1] Ravi Kumar — 6 skills
  [2] Sneha Reddy — 6 skills
  [3] Arun Pillai — 8 skills
  [4] Priya Nair — 5 skills
  [5] Karthik Sharma — 5 skills

5/5 succeeded, 0 failed


## Step 5 — Save jds.jsonl (input for Day 7 RAG)

In [6]:
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print(f'Empty input: {type(e).__name__}: {str(e)[:200]}')

# Whitespace only
try:
    bad = extract_resume('   \n\n   ')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print(f'Whitespace input: {type(e).__name__}: {str(e)[:200]}')

# Garbage non-résumé text
try:
    bad = extract_resume('the quick brown fox jumps over the lazy dog')
    print('Garbage input:', bad.model_dump_json())
except Exception as e:
    print(f'Garbage input: {type(e).__name__}: {str(e)[:200]}')

Unexpected success: {"name":"John Doe","email":"john.doe@example.com","phone":null,"education":[{"degree":"Master of Science in Computer Science","institution":"University of Example","year":2022},{"degree":"Bachelor of Science in Software Engineering","institution":"Another University","year":2020}],"skills":["Python","Java","JavaScript","React","SQL","Cloud Computing (AWS)","Machine Learning"],"projects":["Developed a web-based e-commerce platform","Built a data analytics dashboard using Python and Tableau"],"experience_years":2.5}
Unexpected success: {"name":"","email":"","phone":null,"education":[],"skills":[],"projects":[],"experience_years":0.0}
Garbage input: {"name":"John Doe","email":"john.doe@example.com","phone":null,"education":[{"degree":"Bachelor of Science","institution":"University of Nowhere","year":2020}],"skills":["Problem Solving","Quick Learner"],"projects":[],"experience_years":0.0}


## Step 6 — Engineer Answer (in README)Fill these 5 questions. Push as part of your Day 6 commit.```PROBLEM      — What real placement-training pain are we solving?ARCHITECTURE — Boxes-and-arrows diagram (sketch).TRADE-OFFS   — Cost / latency / accuracy / complexity.SCALE        — What breaks at 10x, 100x, 10000x?INTERVIEW    — 2-sentence summary for an HR panel.```

In [7]:
class JD(BaseModel):
    company: str
    role: str
    must_have_skills: List[str]
    nice_to_have_skills: List[str] = []
    min_cgpa: Optional[float] = None
    locations: List[str] = []
    package_lpa: Optional[float] = None

In [9]:
import requests
from bs4 import BeautifulSoup
import pathlib, json

def fetch_jd(url, max_chars=6000):
    """Fetch JD URL and return clean text. Returns None on block / failure."""
    try:
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'html.parser')
        # Remove script and style tags
        for tag in soup(['script', 'style']):
            tag.decompose()
        return soup.get_text(separator='\n', strip=True)[:max_chars]
    except Exception as e:
        print(f'  Scrape failed for {url}: {e}')
        return None

# Test on one URL
test_url = 'https://amazon.jobs/en/jobs/10407699/sr-bi-engineer-bi-data-engineering-agl-ba-bi'
text = fetch_jd(test_url)
if text:
    print(f'Got {len(text)} chars')
    print(text[:300])
else:
    print('Scrape blocked. Will use cached set.')

Got 4991 chars
Sr. BI Engineer (BI + Data Engineering), AGL BA&BI - Job ID: 10407699 | Amazon.jobs
Skip to main content
×
Home
Teams
Locations
Job categories
My career
My applications
My profile
Account security
Settings
Sign out
Resources
Accommodations
Benefits
Inclusive experiences
How We Hire
Leadership princi


In [10]:
def normalise_jd(text: str) -> JD:
    """Send JD text to Gemini, get structured JD JSON back."""
    resp = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=f'Extract a JD JSON from this text:\n\n{text}',
        config={
            'response_mime_type': 'application/json',
            'response_schema': JD.model_json_schema(),
        },
    )
    return JD.model_validate_json(resp.text)

# Test on one JD text
if text:
    jd = normalise_jd(text)
    print(jd.model_dump_json(indent=2))

{
  "company": "Amazon",
  "role": "Sr. BI Engineer (BI + Data Engineering)",
  "must_have_skills": [
    "10+ years of professional or military experience",
    "5+ years of SQL experience",
    "Experience programming to extract, transform and clean large (multi-TB) data sets",
    "Experience with theory and practice of design of experiments and statistical analysis of results",
    "Experience with AWS technologies",
    "Experience in scripting for automation (e.g. Python) and advanced SQL skills",
    "Experience with theory and practice of information retrieval, data science, machine learning and data mining",
    "Bachelor’s degree in Computer Science, Engineering, Math, Finance, Statistics or related discipline",
    "Use analytical and statistical rigor to solve complex problems and drive business decisions",
    "Manipulate/mine data from database tables (ex. Redshift, MySQL) using SQL",
    "Manipulate/mine data from log files by writing scripts (ex. PERL, Python, UNIX)",
 

In [11]:
import json, pathlib

URLS = [
    # Paste your 5 assigned URLs here
    'https://amazon.jobs/en/jobs/10415299/business-intel-engineer-go-ai',
    'https://amazon.jobs/en/jobs/10415243/business-intelligence-engineer-scot-aim',
    'https://amazon.jobs/en/jobs/10415031/sr-bie-amazon-leo-b2b',
    'https://amazon.jobs/en/jobs/10414716/director-product-analytics',
    'https://amazon.jobs/en/jobs/10407699/sr-bi-engineer-bi-data-engineering-agl-ba-bi',
]

CACHE = pathlib.Path('../data/jds_cached.jsonl')
USE_CACHE = False   # set True if scraping is blocked

jds = []

if USE_CACHE and CACHE.exists():
    print(f'Using cached JDs from {CACHE}')
    for line in CACHE.read_text().splitlines():
        jds.append(JD.model_validate_json(line))
else:
    for url in URLS:
        text = fetch_jd(url)
        if text is None:
            continue
        try:
            jd = normalise_jd(text)
            jds.append(jd)
            print(f'  ✓ {jd.company} — {jd.role}')
        except Exception as e:
            print(f'  ✗ {url}: {e}')

print(f'\nProcessed {len(jds)} JDs')

# Inspect first 3
for jd in jds[:3]:
    print(f'\n{jd.company} - {jd.role}')
    print(f'  Must: {jd.must_have_skills}')
    print(f'  Nice: {jd.nice_to_have_skills}')
    print(f'  CGPA: {jd.min_cgpa}, LPA: {jd.package_lpa}')

  ✓ Amazon — Business Intel Engineer, GO-AI
  ✓ Amazon.com Services LLC — Business Intelligence Engineer, SCOT-AIM
  ✓ Amazon — Sr. BIE, Amazon Leo B2B
  ✓ Audible, Inc. — Director, Product Analytics
  ✓ Amazon — Sr. BI Engineer (BI + Data Engineering)

Processed 5 JDs

Amazon - Business Intel Engineer, GO-AI
  Must: ['Analyzing and interpreting data with Redshift, Oracle, NoSQL', 'SQL', 'ETL', 'Processing large, multi-dimensional datasets from multiple sources', 'Performing statistical analysis', 'Developing automated reporting', 'Data visualization using Tableau, Quicksight, or similar tools', 'Data modeling', 'Data warehousing', 'Building ETL pipelines', 'Statistical Analysis packages such as R, SAS, Matlab', 'Python scripting for data processing', 'Knowledge of Generative AI solutions', 'Excellent verbal and written communication skills']
  Nice: ['AWS solutions (EC2, DynamoDB, S3, Redshift)', 'Data mining', 'Experience using databases with large-scale, complex datasets', 'Working 

In [14]:
OUT = pathlib.Path('./sample_data/jds.jsonl')
OUT.parent.mkdir(exist_ok=True)
with open(OUT, 'w') as f:
    for jd in jds:
        f.write(jd.model_dump_json() + '\n')
print(f'Wrote {len(jds)} JDs to {OUT}')

# Verify the file
with open(OUT) as f:
    for line in f:
        d = json.loads(line)
        print(f'  {d["company"]:20} | {d["role"]:30} | {len(d["must_have_skills"])} must-haves')

Wrote 5 JDs to sample_data/jds.jsonl
  Amazon               | Business Intel Engineer, GO-AI | 14 must-haves
  Amazon.com Services LLC | Business Intelligence Engineer, SCOT-AIM | 15 must-haves
  Amazon               | Sr. BIE, Amazon Leo B2B        | 12 must-haves
  Audible, Inc.        | Director, Product Analytics    | 8 must-haves
  Amazon               | Sr. BI Engineer (BI + Data Engineering) | 11 must-haves
